# MAuto A/B Debate Diagnostic

Notebook này dùng local controller đang chạy tại `http://127.0.0.1:8500` để test live role `A` và `B` trên ChatGPT qua Tampermonkey script đã paste sẵn trong Chrome.

Flow mục tiêu:
- kiểm tra role online
- PROBE + WAIT_COMPOSER_STABLE cho A/B
- chạy 7 lượt tranh luận về kinh tế Việt Nam năm 2026
- A: nêu luận điểm và góp ý
- B: phản biện
- log đầy đủ từng bước gửi prompt, tìm nút, click send, chờ assistant xong

In [ ]:
# BLOCK 1: imports + config
import json
import time
import urllib.request
import urllib.parse
from IPython.display import clear_output

BASE_URL = "http://127.0.0.1:8500"
ROLES = ["A", "B"]
TERMINAL_STATES = {
    "PROBE_DONE",
    "DUMP_BUTTONS_DONE",
    "COMPOSER_STABLE",
    "COMPOSER_UNSTABLE",
    "PASTE_CONFIRMED",
    "PASTE_FAILED",
    "FIND_SEND_DONE",
    "SEND_BUTTON_ENABLED_DONE",
    "SEND_ACCEPTED",
    "SEND_FAILED",
    "ASSISTANT_DONE",
    "ASSISTANT_TIMEOUT",
    "TRANSCRIPT_SAVED",
    "TRANSCRIPT_SAVE_ACK",
    "UNKNOWN_COMMAND",
    "ERROR_COMMAND",
}

print("BASE_URL:", BASE_URL)
print("ROLES:", ROLES)

In [ ]:
# BLOCK 2: low-level HTTP + admin helpers
def http_json(method, path, payload=None):
    url = f"{BASE_URL}{path}"
    data = None
    headers = {}
    if payload is not None:
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = urllib.request.Request(url, data=data, method=method, headers=headers)
    with urllib.request.urlopen(req, timeout=300) as resp:
        return json.loads(resp.read().decode("utf-8"))

def role_snapshot(role):
    return http_json("GET", f"/api/admin/role/{urllib.parse.quote(role)}")

def tail_events(role="", contains="", command_id="", limit=30):
    params = urllib.parse.urlencode({
        "role": role,
        "contains": contains,
        "command_id": command_id,
        "limit": limit,
    })
    return http_json("GET", f"/api/admin/events?{params}")

def send_command(role, action, payload=None):
    data = http_json("POST", "/api/admin/command", {
        "role": role,
        "action": action,
        "payload": payload or {},
    })
    return data["command"]

def wait_command(command_id, print_every=1.0, timeout=300):
    started = time.time()
    last_print = 0
    while True:
        data = http_json("GET", f"/api/admin/command/{command_id}")
        now = time.time()
        if now - last_print >= print_every:
            print(f"[{time.strftime('%H:%M:%S')}] cmd={command_id[:8]} status={data['status']} done={data['done']}")
            last_print = now
        if data["done"]:
            return data["result"]
        if timeout and now - started > timeout:
            raise TimeoutError(f"Timeout waiting for command_id={command_id}")
        time.sleep(0.2)

def run_command(role, action, payload=None, timeout=300, print_every=1.0):
    cmd = send_command(role, action, payload)
    print(f"{action} -> role={role}, command_id={cmd['command_id']}")
    return wait_command(cmd["command_id"], print_every=print_every, timeout=timeout)

def dom_summary(dom):
    messages = dom.get("messages") or {}
    counts = messages.get("counts") if isinstance(messages, dict) else None
    return {
        "composer": bool(dom.get("composer")),
        "composer_text_len": dom.get("composer_text_len"),
        "send_enabled": dom.get("send_enabled"),
        "stop_visible": dom.get("stop_visible"),
        "voice_visible": dom.get("voice_visible"),
        "message_counts": counts,
        "selection_strategy": dom.get("selection_strategy"),
        "matched_selector": dom.get("matched_selector"),
    }

print("Helpers ready")

In [ ]:
# BLOCK 3: wait roles online
for role in ROLES:
    print("=" * 80)
    print("ROLE", role)
    print("=" * 80)
    for i in range(30):
        snap = role_snapshot(role)
        print(f"[{i}] status={snap['status']} sessions={snap['sessions']}")
        if snap["status"] != "OFFLINE" or snap["dom_info"]:
            print("Connected:", json.dumps(dom_summary(snap["dom_info"]), ensure_ascii=False, indent=2))
            break
        time.sleep(1)
    else:
        raise RuntimeError(f"Role {role} is still offline")

In [ ]:
# BLOCK 4: baseline probe + composer stability for A/B
for role in ROLES:
    print("\n" + "=" * 100)
    print(f"PROBE {role}")
    print("=" * 100)
    probe_result = run_command(role, "PROBE", timeout=60)
    print("state:", probe_result["state"])
    print(json.dumps(dom_summary(probe_result.get("dom_info", {})), ensure_ascii=False, indent=2))

    print("\n" + "=" * 100)
    print(f"WAIT_COMPOSER_STABLE {role}")
    print("=" * 100)
    stable_result = run_command(role, "WAIT_COMPOSER_STABLE", {"samples": 10, "sample_ms": 300}, timeout=60)
    print("state:", stable_result["state"])
    print(json.dumps(stable_result.get("result", {}), ensure_ascii=False, indent=2))

In [ ]:
# BLOCK 5: debate personas + helpers
TOPIC = "Kinh tế Việt Nam năm 2026"

A_PERSONA = """Bạn là Agent A. Vai trò của bạn là người phân tích và góp ý chính sách.\n"
A_PERSONA += "Mục tiêu: đưa ra nhận định cân bằng, thực dụng, ưu tiên hành động.\n"
A_PERSONA += "Trong cuộc trao đổi này, bạn mở đầu bằng một luận điểm ngắn về kinh tế Việt Nam 2026, sau đó ở các lượt sau chỉ phản hồi trực tiếp ý gần nhất từ B.\n"
A_PERSONA += "Mỗi câu trả lời dài 120-220 từ, rõ ý, không dùng bullet nếu không cần.\n"
A_PERSONA += "Luôn kèm góp ý chính sách hoặc khuyến nghị thực thi cụ thể."""

B_PERSONA = """Bạn là Agent B. Vai trò của bạn là người phản biện khó tính.\n"
B_PERSONA += "Mục tiêu: tìm lỗ hổng trong lập luận của A, phản biện bằng logic và rủi ro thực thi.\n"
B_PERSONA += "Bạn không công kích cá nhân, chỉ phản biện nội dung.\n"
B_PERSONA += "Mỗi câu trả lời dài 120-220 từ, tập trung phản biện 2-3 điểm mạnh nhất từ phát biểu gần nhất của A."""

DEBATE_RULES = """Chủ đề cố định: Kinh tế Việt Nam năm 2026.\n"
DEBATE_RULES += "A là bên phân tích và góp ý. B là bên phản biện.\n"
DEBATE_RULES += "Giữ giọng điệu chuyên nghiệp, không lan man, không nói về vai trò hệ thống.\n"
DEBATE_RULES += "Luôn trả lời bằng tiếng Việt."""

def set_prompt_and_send(role, prompt_text, timeout=300):
    print("\n" + "-" * 100)
    print(f"SET_PROMPT {role}")
    print("-" * 100)
    r1 = run_command(role, "SET_PROMPT", {
        "text": prompt_text,
        "method": "auto",
        "samples": 6,
        "sample_ms": 300,
    }, timeout=120)
    print("state:", r1["state"])
    print(json.dumps(r1.get("result", {}), ensure_ascii=False, indent=2))

    print("\n" + "-" * 100)
    print(f"FIND_SEND {role}")
    print("-" * 100)
    r2 = run_command(role, "FIND_SEND", timeout=120)
    print("state:", r2["state"])
    print(json.dumps(r2.get("result", {}), ensure_ascii=False, indent=2))

    print("\n" + "-" * 100)
    print(f"CLICK_SEND {role}")
    print("-" * 100)
    r3 = run_command(role, "CLICK_SEND", timeout=120)
    print("state:", r3["state"])
    print(json.dumps(r3.get("result", {}), ensure_ascii=False, indent=2))

    print("\n" + "-" * 100)
    print(f"WAIT_ASSISTANT_DONE {role}")
    print("-" * 100)
    r4 = run_command(role, "WAIT_ASSISTANT_DONE", timeout=timeout)
    print("state:", r4["state"])
    preview = (r4.get("text") or "")[:1200]
    print(preview)
    return {
        "set_prompt": r1,
        "find_send": r2,
        "click_send": r3,
        "assistant": r4,
    }

def build_initial_prompt_for_a():
    return f"{DEBATE_RULES}\n\n{A_PERSONA}\n\nHãy mở đầu cuộc trao đổi. Viết một nhận định cô đọng về triển vọng, rủi ro và 2 khuyến nghị chính sách cho {TOPIC}."

def build_reply_prompt_for_b(a_text, round_no):
    return f"{DEBATE_RULES}\n\n{B_PERSONA}\n\nĐây là phát biểu gần nhất của A ở lượt {round_no}:\n\n{a_text}\n\nHãy phản biện trực tiếp, nêu điểm yếu trong lập luận và rủi ro bị đánh giá thấp."

def build_reply_prompt_for_a(b_text, round_no):
    return f"{DEBATE_RULES}\n\n{A_PERSONA}\n\nĐây là phản biện gần nhất của B ở lượt {round_no}:\n\n{b_text}\n\nHãy đáp lại từng trọng điểm và đưa ra góp ý chính sách/giải pháp khả thi hơn."

print("Debate helpers ready")

In [ ]:
# BLOCK 6: run A/B debate for 7 rebuttal rounds
conversation_log = []

print("=" * 100)
print("A OPENING")
print("=" * 100)
a_turn = set_prompt_and_send("A", build_initial_prompt_for_a(), timeout=300)
a_text = (a_turn["assistant"].get("text") or "").strip()
conversation_log.append({"round": 0, "role": "A", "text": a_text})

for round_no in range(1, 8):
    print("\n" + "#" * 100)
    print(f"ROUND {round_no} - B REBUTTAL")
    print("#" * 100)
    b_turn = set_prompt_and_send("B", build_reply_prompt_for_b(a_text, round_no), timeout=300)
    b_text = (b_turn["assistant"].get("text") or "").strip()
    conversation_log.append({"round": round_no, "role": "B", "text": b_text})

    print("\n" + "#" * 100)
    print(f"ROUND {round_no} - A RESPONSE")
    print("#" * 100)
    a_turn = set_prompt_and_send("A", build_reply_prompt_for_a(b_text, round_no), timeout=300)
    a_text = (a_turn["assistant"].get("text") or "").strip()
    conversation_log.append({"round": round_no, "role": "A", "text": a_text})

print("Debate done. Entries:", len(conversation_log))

In [ ]:
# BLOCK 7: summary + latest DOM snapshots
for item in conversation_log:
    print("=" * 100)
    print(f"ROUND {item['round']} | ROLE {item['role']}")
    print("=" * 100)
    print(item["text"][:3000])
    print()

print("\nLatest snapshots")
for role in ROLES:
    snap = role_snapshot(role)
    print("-" * 100)
    print(role, json.dumps(dom_summary(snap.get("dom_info", {})), ensure_ascii=False, indent=2))

In [ ]:
# BLOCK 8: quick event tail for diagnostics if a step stalls
events = tail_events(limit=60)
for event in events["events"]:
    print(json.dumps(event, ensure_ascii=False)[:3000])